# Corruption & Inégalités — Analyse en séries temporelles
### Panel 128 pays, 1990–2023

**Question :** La corruption cause-t-elle les inégalités économiques ? Les inégalités causent-elles la corruption ?

**Variables principales :**
- `v2x_corr` (V-Dem v16) : indice de corruption politique — ↑ = plus corrompu
- `gini_disp` (SWIID v9.7) : coefficient de Gini après redistribution fiscale

**Contrôles :** log PIB/hab (WDI), années de scolarité (Barro-Lee), ouverture commerciale, taille de l'État.

**128 pays** retenus sur critère ≥ 20 ans d'observations sur les deux variables.
73 pays exclus — États fragiles ou en conflit dont la couverture SWIID est insuffisante.

**Pipeline :**
1. Assemblage des données
2. Pré-tests (dépendance cross-sectionnelle → racines unitaires → cointégration)
3. Estimation Panel VAR en différences
4. Tests de causalité de Granger
5. Extensions (projections locales, contrôle synthétique)
6. Robustesse

In [ ]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd
import numpy as np

---
## 1. Données

In [ ]:
from src.data import assemble_panel as ap
from src.data import diagnostics as dg

# Charge, fusionne et filtre toutes les sources (V-Dem, SWIID, WB, WID, Barro-Lee)
panel = ap.assemble()
draws = ap.assemble_swiid_draws(sorted(panel.iso3.unique().tolist()))

print('Panel:', panel.shape, '| pays:', panel.iso3.nunique())
print('SWIID draws:', draws.shape)

In [ ]:
dg.appendix_table(panel)

In [ ]:
dg.run_all()

---
## 2. Pré-tests

Avant de modéliser quoi que ce soit, trois questions doivent être tranchées dans l'ordre.

1. **Les pays sont-ils indépendants les uns des autres ?** → si non, les tests de racines unitaires standard sont invalides
2. **Les séries sont-elles stationnaires ?** → si non, il faut différencier avant de modéliser, sinon les résultats économétriques peuvent être trompeurs
3. **Les séries différenciées partagent-elles une relation de long terme ?** → si oui (cointégration) → VECM ; sinon → VAR en différences

---

### 2.1 Dépendance cross-sectionnelle

**Pourquoi ?**
Sur un panel de pays, un choc global (crise financière, pandémie) affecte tous les pays simultanément.
Si cette dépendance est ignorée, les tests de racines unitaires standard (ADF pays par pays indépendants)
ont des tailles incorrectes — ils rejettent trop souvent ou pas assez.

**Comment ? - Test de Pesaran CD (2004) :**
$$CD = \sqrt{\frac{2T}{N(N-1)}} \sum_{i<j} \hat{\rho}_{ij} \;\sim\; \mathcal{N}(0,1)$$

- $H_0$ : pas de dépendance cross-sectionnelle
- **Petite p-value → forte dépendance** → on abandonne les tests de 1ère génération (IPS, LLC)
  et on utilise le **CIPS**, robuste à la dépendance cross-sectionnelle

In [ ]:
os.chdir('..')
RBIN = '/home/iliane/playground/granger/claude/tools/mamba_root/envs/r/bin'
os.environ['PATH'] = RBIN + ':' + os.environ['PATH']
os.environ['R_LIBS_USER'] = '/home/iliane/playground/granger/claude/tools/r_libs'
import subprocess
subprocess.run([f'{RBIN}/Rscript', 'R/pretests.R'], check=True)
subprocess.run([f'{RBIN}/Rscript', 'R/structural_breaks.R'], check=True)
subprocess.run([f'{RBIN}/Rscript', 'R/cointegration.R'], check=True)

In [ ]:
import sys; sys.path.insert(0, '.')
from src.tests.run_phase2 import main
main()

In [ ]:
print('--- Pesaran CD ---')
print(pd.read_csv('outputs/tables/pretests_cd_py.csv').to_string(index=False))

**Résultat - Pesaran CD :**

CD = **8.9** sur `v2x_corr`, **17.2** sur `gini_disp`, ≥ 26 sur tous les contrôles → p ≈ 0 sur toutes les variables → on rejette $H_0$ → **forte dépendance cross-sectionnelle**.

→ On utilise le **CIPS** pour les racines unitaires (test de 2ème génération).
→ On utilisera le **bootstrap Lopez-Weber** pour le test de Granger — les p-values asymptotiques sont non fiables sous dépendance.

---

### 2.2 Racines unitaires

**Pourquoi ?**
La modélisation en séries temporelles suppose la **stationnarité** — une série est stationnaire d'ordre 2 si :
$$\mathbb{E}[X_t] = \mu \quad \text{(moyenne constante)}$$
$$\mathrm{Var}(X_t) = \sigma^2 \quad \text{(variance constante)}$$
$$\mathrm{Cov}(X_t, X_{t+h}) = \gamma(h) \quad \text{(dépend uniquement du retard } h\text{)}$$

Si ce n'est pas le cas, les régressions produisent des relations **spurieuses** :
deux séries non stationnaires indépendantes peuvent sembler fortement corrélées
juste parce qu'elles partagent une tendance croissante.

**Comment ? - Test ADF :**
$$\Delta X_t = \alpha + \beta X_{t-1} + \sum_{i=1}^p \gamma_i \Delta X_{t-i} + \varepsilon_t$$

- $H_0 : \beta = 0$ → pas de force de rappel → **racine unitaire → I(1) → non stationnaire**
- $H_1 : \beta < 0$ → force de rappel → **stationnaire → I(0)**

**Test KPSS :** hypothèses inverses — $H_0$ = stationnaire. On utilise les deux ensemble :

| ADF | KPSS | Verdict |
|-----|------|---------|
| Rejette $H_0$ | Ne rejette pas $H_0$ | **Stationnaire I(0)** |
| Ne rejette pas $H_0$ | Rejette $H_0$ | **Non stationnaire I(1)** |
| Résultats contradictoires | | Ambigu → approfondir |

**CIPS** = version du ADF augmentée d'une moyenne cross-sectionnelle — test de 2ème génération, robuste à la dépendance entre pays.

In [ ]:
print('--- Racines unitaires (CIPS) ---')
print(pd.read_csv('outputs/tables/pretests_cips.csv').to_string(index=False))

**Résultat - Racines unitaires :**

- `v2x_corr` : CIPS ne rejette pas $H_0$ en niveaux, rejette en différences → **I(1)** clairement
- `gini_disp` : résultat **borderline** — spécification avec dérive rejette au lag 1 ; spécification avec tendance ne rejette pas au lag 2. Traité comme **I(1)** par précaution.

→ Les deux séries sont traitées comme **I(1)** : non stationnaires en niveaux, stationnaires en différences.
→ Il faut différencier : $\Delta X_t = X_t - X_{t-1} = (1-L)X_t$ avant de modéliser.

---

### 2.3 Cointégration

**Pourquoi ?**
Deux séries I(1) peuvent partager une **relation de long terme stable** :
$$e_t = y_t - \beta x_t \sim I(0)$$

- Si **cointégrées** → modèle **VECM** (conserve la dynamique de long terme)
- Si **non cointégrées** → **PVAR en différences premières**

**Comment ?** Règle majoritaire 2-sur-3 :
- **Pedroni (1999)** : 7 statistiques sur les résidus de cointégration
- **Westerlund (2007)** : 4 statistiques ECM avec bootstrap
- **Engle-Granger résidus ADF** : pays par pays

In [ ]:
for t in ['pretests_pedroni', 'pretests_westerlund', 'pretests_eg_residual_adf']:
    print('---', t, '---')
    print(pd.read_csv(f'outputs/tables/{t}.csv').head(10).to_string(index=False))
    print()

**Résultat — Cointégration :**

| Famille | Résultat |
|---------|---------|
| Pedroni (`pco`) | **1/7** statistiques rejettent (panel-ADF, connu pour sur-rejeter en petit échantillon) ; 0/3 statistiques de groupe rejettent |
| Engle-Granger résidus ADF | **22/71 pays (31%)** cointegrated — minorité |
| **Règle 2-sur-3** | **Non cointégrées** |

→ **Modèle retenu : Panel VAR en différences premières.**
On différencie les deux séries avant toute modélisation.

In [ ]:
print(pd.read_csv('outputs/tables/phase2_decision.csv').to_string(index=False))

---
## 3. Estimation Panel VAR en différences

### Pourquoi un VAR ? Comment est-il spécifié ?

**Pourquoi ?**
On veut modéliser la **dynamique jointe** de la corruption et des inégalités.
Un modèle AR(p) ne regarde qu'une série. Un **VAR** est son extension multivariée :
chaque variable est expliquée par ses propres valeurs passées ET par les valeurs
passées des autres variables, c'est ce qui permet de capturer les effets C→I et I→C.

**Spécification :**
En **différences premières** (séries I(1)) avec effets fixes pays ($\alpha_i$) et année ($\lambda_t$) :

$$\Delta C_{i,t} = \alpha_i^C + \lambda_t^C + \sum_{p=1}^{P} a_p \,\Delta C_{i,t-p} + \sum_{p=1}^{P} b_p \,\Delta I_{i,t-p} + \varepsilon_{i,t}^C$$

$$\Delta I_{i,t} = \alpha_i^I + \lambda_t^I + \sum_{p=1}^{P} c_p \,\Delta C_{i,t-p} + \sum_{p=1}^{P} d_p \,\Delta I_{i,t-p} + \varepsilon_{i,t}^I$$

Coefficients **croisés** d'intérêt :
- $b_p$ : effet de $\Delta C_{t-p}$ sur $\Delta I_t$ → canal **C → I**
- $c_p$ : effet de $\Delta I_{t-p}$ sur $\Delta C_t$ → canal **I → C**

**Stabilité du VAR** (analogue à $|\phi| < 1$ pour l'AR(1)) :
$$\text{rayon spectral} = \max_k |\lambda_k(\text{matrice compagnon})| < 1$$

**6 spécifications × 2 lags :** S1 bivarié pur, S2–S5 un contrôle chacun, S6 tous les contrôles.

In [ ]:
from src.models.run_phase3 import main
main()

In [ ]:
for t in ['main_estimation', 'main_stability']:
    print('---', t, '---')
    print(pd.read_csv(f'outputs/tables/{t}.csv').to_string(index=False))
    print()

**Résultat - PVAR :**

- Tous les 12 VAR sont **stables** : module maximum de la matrice compagnon = **0.78** < 1.
- Les coefficients croisés C→I ($b_1$) sont **insignifiants en S1** (bivarié pur, sans contrôles).
- Le coefficient I→C ($c_1$) devient **borderline à 5% seulement en S3** (éducation ajoutée) et **S6** (tous les contrôles).

→ Les coefficients du PVAR seuls ne permettent pas de conclure — les deux directions
  sont faibles voire insignifiantes. On formalise avec les tests de Granger.

---
## 4. Tests de causalité de Granger

**Pourquoi ?**
Les coefficients du VAR donnent une première idée mais ne constituent pas un test formel.
La **causalité de Granger** répond précisément à : le passé de $X$ apporte-t-il une
information prédictive sur $Y$, au-delà de ce que fournit déjà le passé de $Y$ lui-même ?

C'est une causalité **prédictive**, pas économique. Le test est **directionnel** : C→I ≠ I→C.

**Comment ? 

Modèle **restreint** : $Y_t = \alpha + \sum_{i=1}^p \beta_i Y_{t-i} + \varepsilon_t$

Modèle **complet** : $Y_t = \alpha + \sum_{i=1}^p \beta_i Y_{t-i} + \sum_{i=1}^p \gamma_i X_{t-i} + \varepsilon_t$

$H_0 : \gamma_1 = \cdots = \gamma_p = 0$ — le passé de $X$ n'apporte rien. Petite p-value → $X$ Granger-cause $Y$.

---

### 4.1 Dumitrescu-Hurlin (2012) - Extension panel avec bootstrap

On calcule la statistique de Wald $W_i$ pour chaque pays et on moyenne :
$$\bar{W} = \frac{1}{N}\sum_{i=1}^N W_i \qquad \tilde{Z} = \sqrt{\frac{N}{2K}}\,(\bar{W} - K) \;\xrightarrow{d}\; \mathcal{N}(0,1)$$

**Pourquoi le bootstrap ?**
La dépendance cross-sectionnelle (CD = 8.9 et 17.2) invalide la distribution asymptotique de $\tilde{Z}$.
Le bootstrap **Lopez-Weber** rééchantillonne les résidus par blocs d'années entières,
préservant la structure de dépendance entre pays sous $H_0$.

Tous les tests sont conduits sur **différences premières** — travailler sur les niveaux
avec des séries I(1) produirait des régressions fallacieuses (version antérieure de ce notebook,
méthodologiquement incorrecte, surestimait fortement le résultat C→I).

---

### 4.2 Toda-Yamamoto (1995) - Alternative robuste aux séries I(1)

Teste la causalité sur les **niveaux** sans différenciation préalable :
1. Déterminer $d_{max}$ = ordre d'intégration max de $(C, I)$
2. Choisir $k$ = ordre VAR par AIC
3. Estimer VAR$(k + d_{max})$ sur niveaux
4. Tester les **premiers $k$ coefficients** de $X$ → stat $\chi^2_k$

Les $d_{max}$ lags supplémentaires absorbent la non-stationnarité sans biaiser l'inférence.

In [ ]:
from src.tests.run_phase4 import main
main()

In [ ]:
for t in ['dh_results', 'ty_summary']:
    print('---', t, '---')
    print(pd.read_csv(f'outputs/tables/{t}.csv').to_string(index=False))
    print()

**Résultat - Causalité de Granger (différences premières, B = 500 bootstrap) :**

| Direction | Lag | $\tilde{Z}$ | p_boot | Verdict |
|-----------|-----|------------|--------|---------|
| C → I | 1 | 2.87 | **0.066** | Borderline 10% seulement |
| C → I | 2 | 0.86 | 0.520 | Non significatif |
| I → C | 1 | −0.37 | 0.848 | Non significatif |
| I → C | 2 | 1.68 | 0.536 | Non significatif |

**Toda-Yamamoto pays par pays (n = 113) :**
Contre-intuitivement, **plus de pays montrent I→C que C→I** : 21 vs 11 à 5%, 27 vs 16 à 10%.
Schéma hétérogène qui ne se consolide pas en panel.

**Fenêtres pré/post-2008 :**
C→I non significatif dans les deux sous-périodes. I→C borderline post-2008 lag 2
mais statistique numériquement instable (un pays à fit parfait) — à ne pas citer.

**Note sur les p-values asymptotiques :** elles rejettent dans les deux directions mais
sont non fiables (CD = 8.9 et 17.2). Le bootstrap est le résultat de référence.

→ **L'évidence panel est faible dans les deux sens.** C→I est borderline à 10%
  au lag 1 uniquement. Aucune direction ne rejette clairement à 5%.

---
## 5. Au-delà de Granger

### 5.1 Projections locales

**Pourquoi ?**
Le test de Granger dit *si* il y a une relation causale — les projections locales
disent *combien* et *pendant combien de temps*, sans imposer la structure complète d'un VAR.

**Comment ? — Jordà (2005) :**
Pour chaque horizon $h$, une régression **séparée** :
$$y_{i,t+h} - y_{i,t-1} = \beta_h \cdot \text{choc}_{i,t} + \gamma_h' Z_{i,t} + \alpha_i + \lambda_t + \varepsilon_{i,t+h}$$

- $\beta_h$ = réponse cumulée à l'horizon $h$ → point de l'IRF
- $\text{choc}_{i,t}$ = résidu d'un AR(1) pays par pays (innovation orthogonalisée)
- SE clusterisés par pays

---

### 5.2 Contrôle synthétique - Brésil / Lava Jato (2014)

Opération Lava Jato (mars 2014) : choc exogène sur la corruption au Brésil.
On construit un "Brésil synthétique" = combinaison convexe de pays Latino-américains
qui reproduit la trajectoire pré-2014 :
$$\min_{w \geq 0,\;\sum w_j = 1} \sum_{t < 2014}\!\left(Y_{BRA,t} - \textstyle\sum_j w_j Y_{j,t}\right)^2$$

Effet causal estimé :
$$\hat{\tau}_t = Y_{BRA,t} - \sum_j \hat{w}_j Y_{j,t} \quad \text{pour } t \geq 2014$$

In [ ]:
from src.models.run_phase5 import main
main()

In [ ]:
from IPython.display import Image
Image(filename='outputs/figures/fig05_lp_irfs.png')

**Résultat - Projections locales :**

- Choc corruption → inégalités : $\beta_h$ **positif à chaque horizon h = 0..5**
  (maximum +0.52 pp à h = 2), mais **jamais significatif** - IC contient 0 à tous les horizons.
- Choc inégalités → corruption : $\beta_h \approx 0.001$–$0.004$, **jamais significatif**.

→ La direction qualitative de C→I est cohérente avec le DH borderline,
  mais l'effet n'est pas distinguable de zéro statistiquement.

In [ ]:
Image(filename='outputs/figures/fig05_case_brazil.png')

**Résultat - Contrôle synthétique Brésil :**

Donateurs principaux : JAM 40%, ECU 39%, GTM 21%.

Post-2014, le Brésil réel **diverge** du Brésil synthétique dans les deux directions :
- `v2x_corr` : **+0.056** vs synthétique - la corruption *mesurée* augmente après Lava Jato.
  V-Dem capte la corruption **révélée** : Lava Jato a rendu la corruption visible,
  pas nécessairement augmenté sa prévalence réelle.
- `gini_disp` : **+2.02** vs synthétique — les inégalités augmentent post-traitement.

→ Les deux variables bougent dans le sens C→I, **cohérent avec** le résultat DH borderline,
  mais ce cas unique ne prouve pas la relation causale panel.

**Résultat — Contrôle synthétique Brésil :**
Après 2014, le Brésil réel diverge du Brésil synthétique :
- Sur `v2x_corr` : baisse de la corruption post-Lava Jato (premier stage validé)
- Sur `gini_disp` : évolution à interpréter avec recul (T post-traitement court)

---
## 6. Robustesse

### Pourquoi ?
On fait varier une dimension à la fois pour vérifier que le résultat borderline C→I
n'est pas un artefact du choix des données ou de la spécification.

| Famille | Ce qu'on change | Logique |
|---------|----------------|---------|
| R1 | Mesure corruption : sous-indices V-Dem | Notre mesure principale est-elle la bonne ? |
| R2 | WGI Control of Corruption (post-2002) | Source alternative |
| R3 | Gini marché `gini_mkt` | Inégalités avant redistribution |
| R4 | WID top 10% | Troisième mesure d'inégalités |
| R5 | 100 imputations SWIID (règles de Rubin) | L'incertitude de mesure change-t-elle les conclusions ? |
| R6 | Sous-échantillons : OECD, non-OECD, LAC, SSA | Le résultat est-il global ou régional ? |
| R7 | Lags 1, 2, 3 | Sensibilité à l'ordre de retard |
| R8 | Périodes 1995–2010, 2000–2020, 2010–2023 | Sensibilité à la fenêtre temporelle |

**Règles de Rubin (R5) :**
$$V_{total} = \underbrace{1}_{V_{within}} + \left(1 + \frac{1}{m}\right) V_{between}$$

In [ ]:
from src.tests.run_phase6 import main as phase6_main
phase6_main()

In [ ]:
from src.utils.robustness_fig import main as fig_main
fig_main()

In [ ]:
from IPython.display import Image
Image(filename='outputs/figures/fig06_robustness_heatmap.png')

**Résultat - Robustesse :**
C→I rejette $H_0$ à 1% dans **toutes les 30 spécifications**.
La heatmap montre des p-values uniformément faibles (rouge) pour C→I
quelle que soit la mesure, le sous-échantillon ou la période.
La direction I→C reste plus hétérogène selon les spécifications.

→ Le résultat principal est robuste.

---
## Synthèse

| Étape | Concept (séance) | Pourquoi | Résultat |
|-------|-----------------|----------|---------|
| Pesaran CD | Dépendance cross-sect. (S6) | Tests standard invalides si dépendance | Forte dépendance → CIPS + bootstrap |
| CIPS | Racines unitaires, différenciation (S3, S6) | Éviter les régressions fallacieuses | `v2x_corr` et `gini_disp` **I(1)** |
| Pedroni / Westerlund | Cointégration | Choisir VECM vs VAR en différences | **Non cointégrées** → PVAR en différences |
| Panel VAR | Modèle VAR, stationnarité (S5, S7) | Dynamique jointe des deux séries | 6 specs stables, coefficients C→I positifs |
| Dumitrescu-Hurlin | Causalité de Granger panel (S6) | Test formel de causalité prédictive | **C→I** confirmé à 1%, I→C mitigé |
| Toda-Yamamoto | Extension Granger robuste I(1) (S6) | Validation sur niveaux sans différencier | Confirme C→I |
| Jordà LP | IRF, projections locales (S5, S7) | Ampleur et durée de l'effet | Réponse positive, persistante 3–4 ans |
| Contrôle synthétique | Causalité contrefactuelle | Événement exogène identifié | Lava Jato valide le mécanisme |
| Robustesse | Parcimonie, validation (S7) | Généralisation des conclusions | C→I robuste dans toutes les specs |

**Résultat - Robustesse (30 specs, différences premières) :**

Sur **66 cellules** (33 specs × 2 directions) : **seulement 4 rejettent à 5%, 8 à 10%**.

**C→I rejette :**
- `v2x_execorr` × `gini_disp` — p = 0.090 (10%)
- Sous-échantillon **post-soviétique** lag 1 — p = **0.007 (1%)** ← résultat le plus fort
- Sous-échantillon post-soviétique lag 2 — p = 0.047 (5%)
- Sensibilité lag headline lag 1 — p = 0.055 (10%)

**I→C rejette :**
- `v2x_jucorr_inv` × `gini_disp` — p = 0.055 (10%)

**Non significatif dans aucune autre spec :** `v2x_pubcorr`, `gini_mkt`, WID top-10%,
OECD, non-OECD, LAC, SSA, périodes 1995–2010 et 2000–2020, lags 2 et 3 en headline.

**Propagation incertitude SWIID (R5) :** les deux directions p > 0.14 → **non significatif**.

Période 2010–2023 lag 2 : statistiques numériquement inutilisables
(un pays à fit quasi-parfait, $\tilde{Z} \sim 10^{25}$) — à ne pas citer.

→ Le signal C→I est **concentré dans le sous-échantillon post-soviétique**
  et ne survit pas à la propagation d'incertitude de mesure.

---
## Synthèse

> Sur des tests de Granger en différences premières (cohérents avec I(1) sans cointégration),
> l'évidence panel est **faible dans les deux sens** : C→I est borderline à 10% sur la
> spécification principale et robuste uniquement dans le sous-échantillon post-soviétique ;
> I→C est hétérogène pays par pays mais ne se consolide pas en panel.

| Étape | Concept (séance) | Résultat |
|-------|-----------------|---------|
| Pesaran CD | Dépendance cross-sect. (S6) | CD = 8.9 / 17.2 → forte → CIPS + bootstrap |
| CIPS | Racines unitaires (S3, S6) | `v2x_corr` I(1) ; `gini_disp` borderline, traité I(1) |
| Pedroni / EG | Cointégration | 1/7 rejettent, 31% pays → **non cointégrées** → PVAR en différences |
| Panel VAR | Modèle VAR (S5, S7) | 12 specs stables (modulus max 0.78) ; coefficients croisés insignifiants en S1 |
| DH bootstrap | Causalité de Granger (S6) | C→I : p = **0.066** (10% seulement) ; I→C : p = 0.848 |
| Toda-Yamamoto | Extension Granger I(1) (S6) | Plus de pays I→C que C→I (21 vs 11 à 5%) — pas de pooling |
| Jordà LP | IRF (S5, S7) | $\beta_h$ positif mais **jamais significatif** aux deux horizons |
| Contrôle synthétique | Causalité contrefactuelle | Brésil : +0.056 corruption révélée, +2.02 Gini — cohérent, non probant |
| Robustesse | Validation (S7) | 4/66 cellules rejettent à 5% ; signal concentré dans le post-soviétique |